# Phase 2.5 — Validation Without Human Ratings

Covers 2.5.1-2.5.4 from the plan. 2.5.5 (optional human validation) is skipped per the instructions -- can be added later without restructuring, since inter-model agreement here is its own subsection, not a stand-in for a human-agreement one.

**Structural change from the sketch**: `batch_prompt` originally bundles all 300 validation comments into one prompt per label. Given real-size prompts have already hit timeout issues at much smaller batch sizes earlier in this project, that's chunked here instead -- same checkpointed pattern used for generation scoring.

## 1. Setup

In [1]:
import os
import json
import time
import math
import glob
import asyncio
import random
import itertools

import pandas as pd
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', 200)

OUTPUT_DIR = "/Users/nadia/Desktop/redditRun_june/comment_data/"
ENV_PATH = os.path.join(OUTPUT_DIR, ".env")
VAL_DIR = os.path.join(OUTPUT_DIR, "validation")
os.makedirs(VAL_DIR, exist_ok=True)

TEXT_COL = "body"
ID_COL = "id"
SUBREDDIT_COL = "subreddit_source"   # not "subreddit" -- see earlier column-name notes

loaded = load_dotenv(ENV_PATH)
print(f".env loaded: {loaded}")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
print(f"Key set: {bool(OPENROUTER_API_KEY)}, length: {len(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else 0}")

.env loaded: True
Key set: True, length: 73


## 2. Build `LABELS` (2.4 -- LLooM concepts + deductive taxonomy)

Not built yet elsewhere in this project as actual code -- short enough to include here as a setup step.

In [2]:
# Load your frozen LLooM concepts
frozen_path = os.path.join(OUTPUT_DIR, "frozen_concepts.json")
assert os.path.exists(frozen_path), f"frozen_concepts.json not found at {frozen_path} -- finish 2.3 first."
lloom_concepts = json.load(open(frozen_path))
print(f"Loaded {len(lloom_concepts)} frozen LLooM concepts")

# Deductive taxonomy -- SSBC (Cutrona & Suhr, 1992) + unsupportive branch (Ingram et al., 2001)
TAXONOMY = [
 {"id":"T1","source":"taxonomy","name":"informational_support",
  "prompt":"The commenter provides advice, factual information, instruction, referral to "
           "a source of knowledge, or an assessment of the poster's situation. Includes "
           "suggesting a course of action, explaining how something works, teaching a "
           "skill, or offering an interpretation of what is happening to the poster."},
 {"id":"T2","source":"taxonomy","name":"emotional_support",
  "prompt":"The commenter expresses care, concern, sympathy, understanding, encouragement, "
           "or reassurance. Includes acknowledging the poster's feelings, expressing "
           "sorrow for their situation, or offering comfort."},
 {"id":"T3","source":"taxonomy","name":"esteem_support",
  "prompt":"The commenter validates the poster's worth, competence, or judgment. Includes "
           "expressing confidence in their abilities, telling them their reaction is "
           "reasonable or justified, complimenting them, or relieving them of blame."},
 {"id":"T4","source":"taxonomy","name":"tangible_support",
  "prompt":"The commenter offers concrete assistance or resources. Includes offering to "
           "help directly, offering to review a resume or make an introduction, or "
           "pointing to a specific service, document, template, or tool the poster can use."},
 {"id":"T5","source":"taxonomy","name":"network_support",
  "prompt":"The commenter conveys belonging or connection to others in the same situation. "
           "Includes stating that the poster is not alone, describing the experience as "
           "widely shared among peers, or directing them to a community or group."},
 {"id":"T6","source":"taxonomy","name":"unsupportive_response",
  "prompt":"The commenter minimizes, dismisses, criticizes, blames, or mocks the poster. "
           "Includes stating the problem is normal and not worth raising, attributing it "
           "to the poster's own failings or weakness, or responding with derision."},
]

LABELS = lloom_concepts + TAXONOMY
json.dump(LABELS, open(os.path.join(OUTPUT_DIR, "all_labels.json"), "w"), indent=2)
print(f"{len(LABELS)} labels total ({len(lloom_concepts)} LLooM + {len(TAXONOMY)} taxonomy)")

Loaded 8 frozen LLooM concepts
14 labels total (8 LLooM + 6 taxonomy)


## 3. 2.5.1 -- Draw the validation set

300 comments, stratified by subreddit (60 per subreddit x 5 subreddits). Fixed column name and the `include_groups` issue from earlier.

In [3]:
gen_sample = pd.read_parquet(os.path.join(OUTPUT_DIR, "gen_sample.parquet"))

def sample_subreddit(g):
    sampled = g.sample(min(60, len(g)), random_state=99).copy()
    sampled[SUBREDDIT_COL] = g.name
    return sampled

val = gen_sample.groupby(SUBREDDIT_COL, group_keys=False).apply(sample_subreddit)
val = val.reset_index(drop=True)

print(f"Validation set: {len(val)} comments")
print(val[SUBREDDIT_COL].value_counts())

VAL_PATH = os.path.join(VAL_DIR, "val_set.parquet")
val.to_parquet(VAL_PATH)
print(f"Saved: {VAL_PATH}")

Validation set: 276 comments
subreddit_source
SecurityCareerAdvice    60
asknetsec               60
cybersecurity           60
sysadmin                60
ciso                    36
Name: count, dtype: int64
Saved: /Users/nadia/Desktop/redditRun_june/comment_data/validation/val_set.parquet


/var/folders/41/b55bchyx62j34hmsgfhfk2gw0000gp/T/ipykernel_75673/2155196951.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val = gen_sample.groupby(SUBREDDIT_COL, group_keys=False).apply(sample_subreddit)


## 4. Model setup -- reuse across all 3 judges (OpenRouter routes to all three)

In [4]:
JUDGES = {"gemini": "google/gemini-3.7-flash",
          "claude": "anthropic/claude-sonnet-5",
          "gpt":    "openai/gpt-5"}

def setup_llm_fn(api_key):
    from openai import AsyncOpenAI
    import httpx
    return AsyncOpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        timeout=httpx.Timeout(connect=15.0, read=120.0, write=120.0, pool=120.0),
    )

_client = setup_llm_fn(OPENROUTER_API_KEY)

MAX_RETRIES = 5
BASE_DELAY = 5
MAX_OUTPUT_TOKENS = 4096

async def call_model(model_name, prompt):
    for attempt in range(MAX_RETRIES):
        try:
            res = await _client.chat.completions.create(
                model=model_name,
                temperature=0,
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "You are a careful annotator. Follow the instructions exactly."},
                    {"role": "user", "content": prompt},
                ],
            )
            return res.choices[0].message.content if res and res.choices else None
        except Exception as e:
            err_str = str(e).lower()
            is_credits = "402" in str(e) or "requires more credits" in err_str
            is_last_attempt = attempt == MAX_RETRIES - 1
            is_retryable = (
                "429" in str(e) or "rate limit" in err_str or "timed out" in err_str
                or "timeout" in err_str or "connection" in err_str
            )
            if is_credits:
                print(f"  [402 -- out of credits] {e}")
                return None
            if is_retryable and not is_last_attempt:
                delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 2)
                print(f"  [{type(e).__name__}] retrying in {delay:.1f}s (attempt {attempt+1}/{MAX_RETRIES})...")
                await asyncio.sleep(delay)
                continue
            print(f"  [error, giving up after {attempt+1} attempt(s)]: {e}")
            return None
    return None

### Sanity check all three judges before committing to the full run

Given how many surprises we've had with model availability/auth -- confirm all three actually respond before spending on 300 x N labels x 3 judges.

In [5]:
for judge, model_name in JUDGES.items():
    reply = await call_model(model_name, 'Reply with JSON: {"greeting": "hi"}')
    print(f"{judge} ({model_name}): {reply!r}")

gemini (google/gemini-3.7-flash): '{"greeting": "hi"}'
claude (anthropic/claude-sonnet-5): '{"greeting": "hi"}'
gpt (openai/gpt-5): '{"greeting": "hi"}'


## 5. 2.5.2 -- Score with three model families, chunked + checkpointed

Chunked at `VAL_CHUNK_SIZE` comments per call, not all 300 at once. Each (judge, label, chunk) result is cached to disk immediately -- re-running this cell skips whatever's already done, same pattern as generation scoring.

In [6]:
VAL_CHUNK_SIZE = 25   # comments per batch_prompt call -- start conservative, real prompts are risk-prone

def batch_prompt(label, docs):
    numbered = "\n".join(f"{i+1}. {d}" for i, d in enumerate(docs))
    return (f"Concept: {label['name']}\n"
            f"Criteria: {label['prompt']}\n\n"
            f"For each numbered comment, answer 1 if it matches the concept, 0 if not.\n"
            f'Return only JSON, no explanation: {{"1": 0, "2": 1, ...}}\n\n{numbered}')

def parse_batch_response(text, expected_n):
    try:
        parsed = json.loads(text)
        return {int(k): int(v) for k, v in parsed.items()}
    except Exception as e:
        print(f"    ERROR parsing response: {e}")
        return {}

async def score_label_for_judge(judge, model_name, label, val_df):
    label_name = label["name"]
    ckpt_path = os.path.join(VAL_DIR, f"val_{judge}_{label['id']}.json")
    if os.path.exists(ckpt_path):
        return json.load(open(ckpt_path))

    docs = val_df[TEXT_COL].tolist()
    doc_ids = val_df[ID_COL].tolist()
    n_chunks = math.ceil(len(docs) / VAL_CHUNK_SIZE)

    all_scores = {}  # doc_id -> 0/1
    for chunk_idx in range(n_chunks):
        start = chunk_idx * VAL_CHUNK_SIZE
        end = min(start + VAL_CHUNK_SIZE, len(docs))
        chunk_docs = docs[start:end]
        chunk_ids = doc_ids[start:end]

        prompt = batch_prompt(label, chunk_docs)
        response = await call_model(model_name, prompt)
        if response is None:
            print(f"    [{judge}/{label_name}] chunk {chunk_idx+1}/{n_chunks} FAILED -- leaving as missing")
            continue

        parsed = parse_batch_response(response, len(chunk_docs))
        for local_idx, val_score in parsed.items():
            if 1 <= local_idx <= len(chunk_ids):
                all_scores[chunk_ids[local_idx - 1]] = val_score

    json.dump(all_scores, open(ckpt_path, "w"))
    return all_scores


async def run_all_judges(val_df, labels):
    results = {judge: {} for judge in JUDGES}
    for judge, model_name in JUDGES.items():
        print(f"\n=== Judge: {judge} ({model_name}) ===")
        for label in labels:
            t0 = time.time()
            scores = await score_label_for_judge(judge, model_name, label, val_df)
            n_scored = len(scores)
            print(f"  {label['name']}: {n_scored}/{len(val_df)} scored ({time.time()-t0:.1f}s)")
            results[judge][label["name"]] = scores
    return results

scores = await run_all_judges(val, LABELS)


=== Judge: gemini (google/gemini-3.7-flash) ===
  Workplace Problem Guidance: 276/276 scored (0.0s)
  Career Planning Advice: 276/276 scored (0.0s)
  Emotional Support: 276/276 scored (0.0s)
  Critical Pushback: 276/276 scored (0.0s)
  Personal Relating: 276/276 scored (0.0s)
  Discussion Direction: 276/276 scored (0.0s)
  Practical Advice: 276/276 scored (0.0s)
  Support Connections: 276/276 scored (0.0s)
  informational_support: 276/276 scored (0.0s)
  emotional_support: 276/276 scored (0.0s)
  esteem_support: 276/276 scored (0.0s)
  tangible_support: 276/276 scored (0.0s)
  network_support: 276/276 scored (0.0s)
  unsupportive_response: 276/276 scored (0.0s)

=== Judge: claude (anthropic/claude-sonnet-5) ===
  Workplace Problem Guidance: 276/276 scored (0.0s)
  Career Planning Advice: 276/276 scored (0.0s)
  Emotional Support: 276/276 scored (0.0s)
  Critical Pushback: 276/276 scored (0.0s)
  Personal Relating: 276/276 scored (0.0s)
  Discussion Direction: 276/276 scored (0.0s)
  P

### Save results to `val_scores_{judge}.jsonl` (per your plan's naming)

One line per label, containing that label's full doc_id -> score mapping.

In [7]:
for judge in JUDGES:
    path = os.path.join(VAL_DIR, f"val_scores_{judge}.jsonl")
    with open(path, "w") as f:
        for label_name, doc_scores in scores[judge].items():
            f.write(json.dumps({"label": label_name, "scores": doc_scores}) + "\n")
    print(f"Saved {path}")

Saved /Users/nadia/Desktop/redditRun_june/comment_data/validation/val_scores_gemini.jsonl
Saved /Users/nadia/Desktop/redditRun_june/comment_data/validation/val_scores_claude.jsonl
Saved /Users/nadia/Desktop/redditRun_june/comment_data/validation/val_scores_gpt.jsonl


In [8]:
# network_support is label id "T5" in your TAXONOMY
# bad_ckpt = os.path.join(VAL_DIR, "val_gpt_T5.json")
# if os.path.exists(bad_ckpt):
#     os.remove(bad_ckpt)
#     print(f"Removed {bad_ckpt} -- will redo on next run")
# else:
#     print("Not found -- check the filename matches your actual label id")

In [13]:

! pip install "numpy<2.0,>=1.23" "scipy==1.12.0" "irrCAC==0.4.4"

  Using cached scipy-1.12.0-cp312-cp312-macosx_10_9_x86_64.whl.metadata (60 kB)
Using cached scipy-1.12.0-cp312-cp312-macosx_10_9_x86_64.whl (38.9 MB)

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 6. 2.5.3 -- Compute Gwet's AC1 per label

In [14]:
from irrCAC.raw import CAC

rows = []
for lab in LABELS:
    label_name = lab["name"]

    # Align all 3 judges' scores to the same doc_id order -- missing scores
    # (failed chunks) become NaN and get dropped for that row rather than
    # silently treated as 0, which would bias agreement.
    m = pd.DataFrame({
        j: val[ID_COL].map(scores[j].get(label_name, {}))
        for j in JUDGES
    })
    m = m.dropna()

    if len(m) < 2:
        print(f" {label_name}: too few fully-scored rows ({len(m)}) to compute agreement -- skipping")
        continue

    cac = CAC(m)
    g = cac.gwet()["est"]
    k = cac.fleiss()["est"]
    rows.append({
        "label": label_name, "source": lab["source"], "n_rated": len(m),
        "prevalence": m.mean().mean(),
        "pct_agree": (m.nunique(axis=1) == 1).mean(),
        "AC1": g["coefficient_value"], "AC1_ci": g["confidence_interval"],
        "fleiss_k": k["coefficient_value"],
    })

agree = pd.DataFrame(rows).sort_values("AC1")
agree.to_csv(os.path.join(VAL_DIR, "agreement_results.csv"), index=False)
print(agree)

                         label    source  n_rated  prevalence  pct_agree  \
7          Support Connections     lloom      276    0.248792   0.731884   
2            Emotional Support     lloom      276    0.322464   0.786232   
6             Practical Advice     lloom      276    0.599034   0.829710   
9            emotional_support  taxonomy      276    0.274155   0.811594   
3            Critical Pushback     lloom      276    0.318841   0.826087   
10              esteem_support  taxonomy      276    0.211353   0.815217   
12             network_support  taxonomy      276    0.193237   0.822464   
1       Career Planning Advice     lloom      276    0.413043   0.880435   
4            Personal Relating     lloom      276    0.341787   0.873188   
0   Workplace Problem Guidance     lloom      276    0.157005   0.862319   
8        informational_support  taxonomy      276    0.794686   0.876812   
11            tangible_support  taxonomy      276    0.153382   0.876812   
5         Di

### Apply the decision rule

AC1 < 0.60 -> rewrite criteria and rerun. Still < 0.60 after one rewrite -> drop the label.

In [15]:
below_threshold = agree[agree["AC1"] < 0.60]
print(f"{len(below_threshold)} of {len(agree)} labels below AC1 = 0.60:\n")
print(below_threshold[["label", "source", "AC1", "prevalence", "n_rated"]])

0 of 14 labels below AC1 = 0.60:

Empty DataFrame
Columns: [label, source, AC1, prevalence, n_rated]
Index: []


In [16]:
print(agree.sort_values("AC1"))

                         label    source  n_rated  prevalence  pct_agree  \
7          Support Connections     lloom      276    0.248792   0.731884   
2            Emotional Support     lloom      276    0.322464   0.786232   
6             Practical Advice     lloom      276    0.599034   0.829710   
9            emotional_support  taxonomy      276    0.274155   0.811594   
3            Critical Pushback     lloom      276    0.318841   0.826087   
10              esteem_support  taxonomy      276    0.211353   0.815217   
12             network_support  taxonomy      276    0.193237   0.822464   
1       Career Planning Advice     lloom      276    0.413043   0.880435   
4            Personal Relating     lloom      276    0.341787   0.873188   
0   Workplace Problem Guidance     lloom      276    0.157005   0.862319   
8        informational_support  taxonomy      276    0.794686   0.876812   
11            tangible_support  taxonomy      276    0.153382   0.876812   
5         Di

## 7. 2.5.4 -- Prompt stability check

Rerun one judge with lightly rephrased criteria and reversed comment order. Report the share of labels where a comment's answer flips.

In [17]:
STABILITY_JUDGE = "gemini"   # cheapest of the three -- fine for a robustness check, not the main result

def perturb(p):
    return "Decide whether each comment matches this description. " + p

async def score_label_perturbed(judge, model_name, label, val_df):
    perturbed_label = {**label, "prompt": perturb(label["prompt"])}
    docs = val_df[TEXT_COL].tolist()[::-1]        # reversed order
    doc_ids = val_df[ID_COL].tolist()[::-1]

    n_chunks = math.ceil(len(docs) / VAL_CHUNK_SIZE)
    all_scores = {}
    for chunk_idx in range(n_chunks):
        start = chunk_idx * VAL_CHUNK_SIZE
        end = min(start + VAL_CHUNK_SIZE, len(docs))
        chunk_docs = docs[start:end]
        chunk_ids = doc_ids[start:end]

        prompt = batch_prompt(perturbed_label, chunk_docs)
        response = await call_model(model_name, prompt)
        if response is None:
            continue
        parsed = parse_batch_response(response, len(chunk_docs))
        for local_idx, val_score in parsed.items():
            if 1 <= local_idx <= len(chunk_ids):
                all_scores[chunk_ids[local_idx - 1]] = val_score
    return all_scores


flip_rows = []
for lab in LABELS:
    label_name = lab["name"]
    original = scores[STABILITY_JUDGE].get(label_name, {})
    ckpt_path = os.path.join(VAL_DIR, f"val_{STABILITY_JUDGE}_{lab['id']}_perturbed.json")
    if os.path.exists(ckpt_path):
        perturbed = json.load(open(ckpt_path))
    else:
        perturbed = await score_label_perturbed(STABILITY_JUDGE, JUDGES[STABILITY_JUDGE], lab, val)
        json.dump(perturbed, open(ckpt_path, "w"))

    common_ids = set(original) & set(perturbed)
    n_flipped = sum(1 for doc_id in common_ids if original[doc_id] != perturbed[doc_id])
    flip_rate = n_flipped / len(common_ids) if common_ids else float("nan")
    flip_rows.append({"label": label_name, "n_compared": len(common_ids),
                       "n_flipped": n_flipped, "flip_rate": flip_rate})

flips = pd.DataFrame(flip_rows).sort_values("flip_rate", ascending=False)
flips.to_csv(os.path.join(VAL_DIR, "prompt_stability.csv"), index=False)
print(flips)

overall_flip_rate = flips["n_flipped"].sum() / flips["n_compared"].sum()
print(f"\nOverall flip rate: {overall_flip_rate:.1%}")
if overall_flip_rate > 0.05:
    print("⚠️  Exceeds ~5% -- measurement is prompt-sensitive; note in limitations.")
else:
    print("✅ Under ~5% -- measurement looks reasonably prompt-stable.")

                         label  n_compared  n_flipped  flip_rate
4            Personal Relating         276         18   0.065217
6             Practical Advice         276         18   0.065217
3            Critical Pushback         276         14   0.050725
9            emotional_support         276         13   0.047101
2            Emotional Support         276         11   0.039855
8        informational_support         276         11   0.039855
1       Career Planning Advice         276          9   0.032609
10              esteem_support         276          8   0.028986
7          Support Connections         276          7   0.025362
12             network_support         276          6   0.021739
0   Workplace Problem Guidance         276          5   0.018116
11            tangible_support         276          4   0.014493
13       unsupportive_response         276          4   0.014493
5         Discussion Direction         276          2   0.007246

Overall flip rate: 3.4%


In [19]:
# from section 2.6 checking here
import json

path = "/Users/nadia/Desktop/redditRun_june/comment_data/score_results/scores.jsonl"
total, empty = 0, 0
by_label = {}

with open(path) as f:
    for line in f:
        row = json.loads(line)
        total += 1
        if not row["scores"]:
            empty += 1
            by_label[row["label_name"]] = by_label.get(row["label_name"], 0) + 1

print(f"Total batches: {total:,}")
print(f"Empty/failed batches: {empty:,} ({empty/total:.1%})")
print("\nBy label:")
for name, count in sorted(by_label.items(), key=lambda x: -x[1]):
    print(f"  {name}: {count}")

Total batches: 74,606
Empty/failed batches: 59,111 (79.2%)

By label:
  Personal Relating: 5329
  Discussion Direction: 5329
  Practical Advice: 5329
  Support Connections: 5329
  informational_support: 5329
  emotional_support: 5329
  esteem_support: 5329
  tangible_support: 5329
  network_support: 5329
  unsupportive_response: 5329
  Critical Pushback: 5328
  Emotional Support: 490
  Workplace Problem Guidance: 2
  Career Planning Advice: 1
